In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------------------------------
# 0. Reload clean data
# ----------------------------------------
train = pd.read_parquet('../data/model/final_train.parquet')
val   = pd.read_parquet('../data/model/final_val.parquet')
test  = pd.read_parquet('../data/model/final_test.parquet')

train = train.sort_values(["tic", "Date"]).reset_index(drop=True)
val   = val.sort_values(["tic", "Date"]).reset_index(drop=True)
test  = test.sort_values(["tic", "Date"]).reset_index(drop=True)

news_features = ["mean_sentiment", "max_sentiment", "min_sentiment",
                 "sum_sentiment", "news_count"]


Using device: cuda


In [2]:
def add_target(df):
    df = df.sort_values(["tic", "Date"]).copy()
    grp_close = df.groupby("tic")["Close"]
    df["ret_1d_raw"] = grp_close.pct_change(1)
    # target = next-day return
    df["target_1d"] = df.groupby("tic")["ret_1d_raw"].shift(-1)
    return df

train = add_target(train)
val   = add_target(val)
test  = add_target(test)

def add_price_features(df):
    df = df.sort_values(["tic","Date"]).copy()

    grp_close = df.groupby("tic")["Close"]
    grp_ret   = df.groupby("tic")["ret_1d_raw"]
    grp_vol   = df.groupby("tic")["Volume"]

    # Price momentum
    df["ret_1d_lag"] = grp_close.pct_change(1)
    df["ret_2d"]     = grp_close.pct_change(2)
    df["ret_3d"]     = grp_close.pct_change(3)
    df["ret_5d"]     = grp_close.pct_change(5)
    df["ret_10d"]    = grp_close.pct_change(10)

    # Intraday + overnight
    prev_close = grp_close.shift(1)
    df["overnight_ret"] = df["Open"] / prev_close - 1
    df["intraday_ret"]  = df["Close"] / df["Open"] - 1

    # RSI
    delta = grp_close.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    roll_up   = gain.groupby(df["tic"]).rolling(14).mean().reset_index(0, drop=True)
    roll_down = loss.groupby(df["tic"]).rolling(14).mean().reset_index(0, drop=True)
    df["RSI"] = 100 - (100 / (1 + roll_up / roll_down))

    # Volatility
    df["vol_3d"]  = grp_ret.rolling(3).std().reset_index(0, drop=True)
    df["vol_5d"]  = grp_ret.rolling(5).std().reset_index(0, drop=True)
    df["vol_10d"] = grp_ret.rolling(10).std().reset_index(0, drop=True)

    # High-Low range volatility
    df["range_vol"] = np.log(df["High"] / df["Low"]) ** 2

    # Liquidity
    mean_vol = grp_vol.transform("mean")
    std_vol  = grp_vol.transform("std")
    df["turnover"] = df["Volume"] / mean_vol
    df["vol_z"]    = (df["Volume"] - mean_vol) / std_vol
    df["amihud"]   = (np.abs(df["ret_1d_raw"]) / df["Volume"]).replace(np.inf, np.nan)

    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df

train = add_price_features(train)
val   = add_price_features(val)
test  = add_price_features(test)

# We’ll just use same-day sentiment (no lag/roll for now to keep LSTM size reasonable)
def add_news_features(df):
    df = df.sort_values(["tic","Date"]).copy()
    # news_features already exist in df from your preprocess
    return df.fillna(0.0)

train = add_news_features(train)
val   = add_news_features(val)
test  = add_news_features(test)

# Drop rows with missing target
train = train[~train["target_1d"].isna()].copy()
val   = val[~val["target_1d"].isna()].copy()
test  = test[~test["target_1d"].isna()].copy()

y_train = train["target_1d"].values
y_val   = val["target_1d"].values
y_test  = test["target_1d"].values

price_features = [
    "ret_1d_lag", "ret_2d", "ret_3d", "ret_5d", "ret_10d",
    "overnight_ret", "intraday_ret",
    "RSI",
    "vol_3d", "vol_5d", "vol_10d",
    "range_vol",
    "turnover", "vol_z", "amihud"
]

feature_cols = price_features + news_features
print("Num features:", len(feature_cols))


Num features: 20


In [3]:
# Standardize features using TRAIN only
feat_mean = train[feature_cols].mean()
feat_std  = train[feature_cols].std().replace(0, 1.0)

def standardize(df):
    df = df.copy()
    df[feature_cols] = (df[feature_cols] - feat_mean) / feat_std
    df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], 0.0).fillna(0.0)
    return df

train = standardize(train)
val   = standardize(val)
test  = standardize(test)


In [4]:
SEQ_LEN = 30   # past 30 days to predict next day

def make_lstm_arrays(df, feature_cols, target_col, seq_len=30):
    X_list = []
    y_list = []

    for tic, g in df.groupby("tic"):
        g = g.sort_values("Date")
        feats = g[feature_cols].values
        targets = g[target_col].values

        for i in range(seq_len - 1, len(g)):
            y = targets[i]
            if np.isnan(y):
                continue
            window = feats[i - seq_len + 1 : i + 1]   # shape (seq_len, n_features)
            X_list.append(window.astype(np.float32))
            y_list.append(np.float32(y))

    X = np.stack(X_list)   # (N, seq_len, n_features)
    y = np.array(y_list, dtype=np.float32)
    return X, y

X_train, y_train_seq = make_lstm_arrays(train, feature_cols, "target_1d", SEQ_LEN)
X_val,   y_val_seq   = make_lstm_arrays(val,   feature_cols, "target_1d", SEQ_LEN)
X_test,  y_test_seq  = make_lstm_arrays(test,  feature_cols, "target_1d", SEQ_LEN)

print("Train sequences:", X_train.shape, y_train_seq.shape)
print("Val   sequences:", X_val.shape,   y_val_seq.shape)
print("Test  sequences:", X_test.shape,  y_test_seq.shape)


Train sequences: (715275, 30, 20) (715275,)
Val   sequences: (110323, 30, 20) (110323,)
Test  sequences: (110227, 30, 20) (110227,)


In [5]:
BATCH_SIZE = 256

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train_seq))
val_ds   = TensorDataset(torch.from_numpy(X_val),   torch.from_numpy(y_val_seq))
test_ds  = TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test_seq))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)


In [6]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        out, (h_n, c_n) = self.lstm(x)
        # use last hidden state
        last_hidden = h_n[-1]          # (batch, hidden_dim)
        out = self.fc(last_hidden)     # (batch, 1)
        return out.squeeze(-1)

input_dim = len(feature_cols)
model = LSTMRegressor(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout=0.2).to(device)
print(model)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


LSTMRegressor(
  (lstm): LSTM(20, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [7]:
import math
best_val_loss = float("inf")
best_state = None
EPOCHS = 20
PATIENCE = 3
no_improve = 0

for epoch in range(1, EPOCHS+1):
    # ---- train ----
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    # ---- validate ----
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch:02d} | Train MSE: {train_loss:.6e} | Val MSE: {val_loss:.6e}")

    # early stopping
    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_state = model.state_dict()
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping.")
            break

# load best weights
if best_state is not None:
    model.load_state_dict(best_state)


Epoch 01 | Train MSE: 4.169393e-04 | Val MSE: 6.044831e-04
Epoch 02 | Train MSE: 3.829142e-04 | Val MSE: 6.203786e-04
Epoch 03 | Train MSE: 3.655604e-04 | Val MSE: 6.080633e-04
Epoch 04 | Train MSE: 3.528301e-04 | Val MSE: 6.237329e-04
Early stopping.


In [8]:
model.eval()
all_preds = []
all_true  = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        preds = model(xb)
        all_preds.append(preds.cpu().numpy())
        all_true.append(yb.numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_true)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
da   = (np.sign(y_pred) == np.sign(y_true)).mean()
ic   = spearmanr(y_pred, y_true).correlation

print("\n=== LSTM (30-day window, price + news) ===")
print("Test RMSE:", rmse)
print("Test DA:  ", da)
print("Test IC:  ", ic)



=== LSTM (30-day window, price + news) ===
Test RMSE: 0.01824440374303085
Test DA:   0.48333892784889365
Test IC:   -0.002386189061873363


In [7]:
# ================================================================
#  TCN MODEL FOR NEXT-DAY RETURN
#  Uses: price_features + news_features + emb_cols
# ================================================================

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------------
# 1. Feature set for TCN (all modalities)
# -------------------------------------------------------

emb_cols = [c for c in train.columns if c.startswith("pca_emb_")]
print("Embedding columns:", len(emb_cols))
tcn_features = price_features + news_features + emb_cols
print("Num TCN features:", len(tcn_features))

SEQ_LEN = 30   # past 30 days to predict next-day return

# -------------------------------------------------------
# 2. Standardize TCN features using TRAIN ONLY
# -------------------------------------------------------
tcn_mean = train[tcn_features].mean()
tcn_std  = train[tcn_features].std().replace(0, 1.0)

def standardize_tcn(df):
    df = df.copy()
    df[tcn_features] = (df[tcn_features] - tcn_mean) / tcn_std
    df[tcn_features] = df[tcn_features].replace([np.inf, -np.inf], 0).fillna(0.0)
    return df

train_tcn = standardize_tcn(train)
val_tcn   = standardize_tcn(val)
test_tcn  = standardize_tcn(test)

# -------------------------------------------------------
# 3. Build sequence arrays (per stock, rolling window)
# -------------------------------------------------------
def make_seq_arrays(df, feature_cols, target_col, seq_len=30):
    X_list, y_list = [], []

    for tic, g in df.groupby("tic"):
        g = g.sort_values("Date")
        feats = g[feature_cols].values
        targets = g[target_col].values

        for i in range(seq_len, len(g)):
            y = targets[i]
            if np.isnan(y):
                continue
            window = feats[i - seq_len:i]  # (seq_len, n_features) - all past days
            X_list.append(window.astype(np.float32))
            y_list.append(np.float32(y))

    X = np.stack(X_list)  # (N, seq_len, n_features)
    y = np.array(y_list, dtype=np.float32)
    return X, y

X_train_tcn, y_train_tcn = make_seq_arrays(train_tcn, tcn_features, "target_1d", SEQ_LEN)
X_val_tcn,   y_val_tcn   = make_seq_arrays(val_tcn,   tcn_features, "target_1d", SEQ_LEN)
X_test_tcn,  y_test_tcn  = make_seq_arrays(test_tcn,  tcn_features, "target_1d", SEQ_LEN)

print("Train TCN sequences:", X_train_tcn.shape, y_train_tcn.shape)
print("Val   TCN sequences:", X_val_tcn.shape,   y_val_tcn.shape)
print("Test  TCN sequences:", X_test_tcn.shape,  y_test_tcn.shape)

# -------------------------------------------------------
# 4. DataLoaders
# -------------------------------------------------------
BATCH_SIZE = 256

train_ds_tcn = TensorDataset(torch.from_numpy(X_train_tcn), torch.from_numpy(y_train_tcn))
val_ds_tcn   = TensorDataset(torch.from_numpy(X_val_tcn),   torch.from_numpy(y_val_tcn))
test_ds_tcn  = TensorDataset(torch.from_numpy(X_test_tcn),  torch.from_numpy(y_test_tcn))

train_loader_tcn = DataLoader(train_ds_tcn, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader_tcn   = DataLoader(val_ds_tcn,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_tcn  = DataLoader(test_ds_tcn,  batch_size=BATCH_SIZE, shuffle=False)

# -------------------------------------------------------
# 5. TCN building blocks
# -------------------------------------------------------
class Chomp1d(nn.Module):
    """Trim off extra padding at the end to keep seq length."""
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        # x: (B, C, L + padding) -> (B, C, L)
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels,
                               kernel_size=kernel_size,
                               padding=pad,
                               dilation=dilation)
        self.chomp1 = Chomp1d(pad)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_channels, out_channels,
                               kernel_size=kernel_size,
                               padding=pad,
                               dilation=dilation)
        self.chomp2 = Chomp1d(pad)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        # 1x1 conv for residual if in/out dims differ
        self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1) \
                          if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class TCNRegressor(nn.Module):
    def __init__(self, input_dim, num_channels=[64, 64, 64], kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        in_ch = input_dim
        for i, out_ch in enumerate(num_channels):
            dilation = 2 ** i
            layers.append(
                TemporalBlock(in_ch, out_ch, kernel_size, dilation, dropout)
            )
            in_ch = out_ch

        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], 1)

    def forward(self, x):
        # x: (B, seq_len, input_dim) -> (B, input_dim, seq_len)
        x = x.transpose(1, 2)
        y = self.tcn(x)            # (B, C, L)
        y = y.mean(dim=2)          # global average pool over time: (B, C)
        out = self.fc(y).squeeze(-1)
        return out


model_tcn = TCNRegressor(input_dim=len(tcn_features),
                         num_channels=[64, 64, 64],
                         kernel_size=3,
                         dropout=0.2).to(device)

print(model_tcn)

criterion_tcn = nn.MSELoss()
optimizer_tcn = torch.optim.Adam(model_tcn.parameters(), lr=1e-3)

# -------------------------------------------------------
# 6. Train TCN with early stopping
# -------------------------------------------------------
EPOCHS = 20
PATIENCE = 3
best_val = float("inf")
best_state = None
pat = 0

for epoch in range(1, EPOCHS + 1):
    # ---- TRAIN ----
    model_tcn.train()
    train_loss = 0.0
    for xb, yb in train_loader_tcn:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer_tcn.zero_grad()
        preds = model_tcn(xb)
        loss = criterion_tcn(preds, yb)
        loss.backward()
        optimizer_tcn.step()

        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader_tcn.dataset)

    # ---- VALID ----
    model_tcn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader_tcn:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model_tcn(xb)
            loss = criterion_tcn(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(val_loader_tcn.dataset)

    print(f"Epoch {epoch:02d} | Train MSE: {train_loss:.6e} | Val MSE: {val_loss:.6e}")

    if val_loss + 1e-6 < best_val:
        best_val = val_loss
        best_state = model_tcn.state_dict()
        pat = 0
    else:
        pat += 1
        if pat >= PATIENCE:
            print("Early stopping.")
            break

if best_state is not None:
    model_tcn.load_state_dict(best_state)

# -------------------------------------------------------
# 7. Evaluate on TEST set
# -------------------------------------------------------
model_tcn.eval()
all_preds = []
all_true  = []

with torch.no_grad():
    for xb, yb in test_loader_tcn:
        xb = xb.to(device)
        preds = model_tcn(xb).cpu().numpy()
        all_preds.append(preds)
        all_true.append(yb.numpy())

y_pred_tcn = np.concatenate(all_preds)
y_true_tcn = np.concatenate(all_true)

tcn_rmse = np.sqrt(mean_squared_error(y_true_tcn, y_pred_tcn))
tcn_da   = (np.sign(y_pred_tcn) == np.sign(y_true_tcn)).mean()
tcn_ic   = spearmanr(y_pred_tcn, y_true_tcn).correlation

print("\n==============================")
print("         TCN RESULTS")
print("==============================")
print("Test RMSE:", tcn_rmse)
print("Test DA:  ", tcn_da)
print("Test IC:  ", tcn_ic)


Using device: cuda
Embedding columns: 64
Num TCN features: 84
Train TCN sequences: (714779, 30, 84) (714779,)
Val   TCN sequences: (109826, 30, 84) (109826,)
Test  TCN sequences: (109727, 30, 84) (109727,)
TCNRegressor(
  (tcn): Sequential(
    (0): TemporalBlock(
      (conv1): Conv1d(84, 64, kernel_size=(3,), stride=(1,), padding=(2,))
      (chomp1): Chomp1d()
      (relu1): ReLU()
      (dropout1): Dropout(p=0.2, inplace=False)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,))
      (chomp2): Chomp1d()
      (relu2): ReLU()
      (dropout2): Dropout(p=0.2, inplace=False)
      (downsample): Conv1d(84, 64, kernel_size=(1,), stride=(1,))
      (relu): ReLU()
    )
    (1): TemporalBlock(
      (conv1): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (chomp1): Chomp1d()
      (relu1): ReLU()
      (dropout1): Dropout(p=0.2, inplace=False)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))


In [11]:
train

,Date,tic,Open,High,Low,Close,Volume,sales_growth_qoq,sales_growth_ttm,asset_growth,...,overnight_ret,intraday_ret,RSI,vol_3d,vol_5d,vol_10d,range_vol,turnover,vol_z,amihud
0,2016-01-04,A,37.978592,38.098833,37.312624,37.636356,3287300,0.020710,-0.002470,-0.309482,...,-0.049343,-0.548922,-3.048453,-1.004015,-1.156413,-1.301821,-0.147776,0.775243,1.182817,-0.017404
1,2016-01-05,A,37.673370,37.876861,37.312638,37.506878,2587200,0.020710,-0.002470,-0.309482,...,0.030081,-0.277111,-3.048453,-1.004015,-1.156413,-1.301821,-0.221039,0.344315,0.525334,-0.017175
2,2016-01-06,A,37.220145,37.913860,37.044401,37.673370,2103600,0.020710,-0.002470,-0.309482,...,-0.666738,0.705276,-3.048453,-1.004015,-1.156413,-1.301821,-0.111620,0.046648,0.071172,-0.017041
3,2016-01-07,A,37.127663,37.136914,35.897475,36.073215,3504300,0.020710,-0.002470,-0.309482,...,-1.219172,-1.696640,-3.048453,0.660353,-1.156413,-1.301821,0.103048,0.908812,1.386608,-0.015319
4,2016-01-08,A,36.276692,36.729917,35.582976,35.693970,3736700,0.020710,-0.002470,-0.309482,...,0.406196,-0.966353,-3.048453,0.583722,-1.156413,-1.301821,0.052075,1.051859,1.604861,-0.016920
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
729654,2021-12-23,ZTS,232.351139,234.937094,231.380211,233.129807,1172400,0.022051,0.163387,-0.001457,...,-0.005886,0.182857,1.706472,-0.555146,-0.648582,-0.502039,-0.218424,-0.655428,-1.001395,-0.016833
729655,2021-12-27,ZTS,234.937073,237.176951,233.677745,236.975067,807000,0.022051,0.163387,-0.001457,...,0.576725,0.497966,1.875781,-0.521111,-0.594368,-0.456293,-0.222552,-0.840166,-1.283647,-0.013888
729656,2021-12-28,ZTS,237.446137,238.253642,234.216099,234.802505,1004400,0.022051,0.163387,-0.001457,...,0.111196,-0.674550,1.459362,-0.153891,-0.357343,-0.374620,-0.197659,-0.740365,-1.131166,-0.015834
729657,2021-12-29,ZTS,234.581391,238.263250,234.216085,237.474960,939900,0.022051,0.163387,-0.001457,...,-0.125395,0.714637,1.415303,-0.104125,-0.381076,-0.623401,-0.197175,-0.772975,-1.180989,-0.015321


In [8]:
# ================================================================
#  TCN WITH TICKER EMBEDDINGS
# ================================================================

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# -------------------------------------------------------
# 1. Ticker → integer ID mapping
# -------------------------------------------------------
all_tickers = sorted(train["tic"].unique())
tic2id = {tic: i for i, tic in enumerate(all_tickers)}
num_tickers = len(tic2id)
print("Tickers:", num_tickers)

# Add numeric ID to dfs
train_te = train.copy()
val_te   = val.copy()
test_te  = test.copy()

train_te["tic_id"] = train_te["tic"].map(tic2id)
val_te["tic_id"]   = val_te["tic"].map(tic2id)
test_te["tic_id"]  = test_te["tic"].map(tic2id)

# -------------------------------------------------------
# 2. Build sequences (features + ticker ID)
# -------------------------------------------------------
def make_lstm_arrays_with_tic(df, feature_cols, target_col, seq_len=30):
    X_list, y_list, tic_list = [], [], []

    for tic, g in df.groupby("tic"):
        g = g.sort_values("Date")
        feats = g[feature_cols].values
        tic_ids = g["tic_id"].values
        targets = g[target_col].values

        for i in range(seq_len, len(g)):
            y = targets[i]
            if np.isnan(y):
                continue
            # past seq_len days of features
            X_list.append(feats[i-seq_len:i].astype(np.float32))
            # ticker id (same for entire sequence)
            tic_list.append(np.int64(tic_ids[i]))
            y_list.append(np.float32(y))

    X = np.stack(X_list)
    y = np.array(y_list, dtype=np.float32)
    tic_ids = np.array(tic_list, dtype=np.int64)
    return X, y, tic_ids

X_train_te, y_train_te, tic_train_ids = make_lstm_arrays_with_tic(
    train_te, tcn_features, "target_1d", SEQ_LEN)
X_val_te,   y_val_te,   tic_val_ids = make_lstm_arrays_with_tic(
    val_te,   tcn_features, "target_1d", SEQ_LEN)
X_test_te,  y_test_te,  tic_test_ids = make_lstm_arrays_with_tic(
    test_te,  tcn_features, "target_1d", SEQ_LEN)

print("Train seq:", X_train_te.shape, y_train_te.shape)
print("Val seq:  ", X_val_te.shape,   y_val_te.shape)
print("Test seq: ", X_test_te.shape,  y_test_te.shape)

# -------------------------------------------------------
# 3. DataLoaders
# -------------------------------------------------------
train_ds_te = TensorDataset(
    torch.from_numpy(X_train_te),
    torch.from_numpy(y_train_te),
    torch.from_numpy(tic_train_ids)
)
val_ds_te = TensorDataset(
    torch.from_numpy(X_val_te),
    torch.from_numpy(y_val_te),
    torch.from_numpy(tic_val_ids)
)
test_ds_te = TensorDataset(
    torch.from_numpy(X_test_te),
    torch.from_numpy(y_test_te),
    torch.from_numpy(tic_test_ids)
)

train_loader_te = DataLoader(train_ds_te, batch_size=256, shuffle=True, drop_last=True)
val_loader_te   = DataLoader(val_ds_te, batch_size=256, shuffle=False)
test_loader_te  = DataLoader(test_ds_te, batch_size=256, shuffle=False)

# -------------------------------------------------------
# 4. TCN block (same as before)
# -------------------------------------------------------
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size]

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, dilation, dropout=0.2):
        super().__init__()
        pad = (k - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=k,
                               padding=pad, dilation=dilation)
        self.chomp1 = Chomp1d(pad)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=k,
                               padding=pad, dilation=dilation)
        self.chomp2 = Chomp1d(pad)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.down = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else None
        self.relu = nn.ReLU()

    def forward(self, x):
        y = self.conv1(x)
        y = self.chomp1(y)
        y = self.relu1(y)
        y = self.drop1(y)

        y = self.conv2(y)
        y = self.chomp2(y)
        y = self.relu2(y)
        y = self.drop2(y)

        res = x if self.down is None else self.down(x)
        return self.relu(y + res)

# -------------------------------------------------------
# 5. TCN model with ticker embedding
# -------------------------------------------------------
class TCNwithTicker(nn.Module):
    def __init__(self, input_dim, num_tickers,
                 emb_dim=8,    # embedding size
                 channels=[64,64,64],
                 kernel=3,
                 dropout=0.2):
        super().__init__()

        self.emb = nn.Embedding(num_tickers, emb_dim)

        layers = []
        in_ch = input_dim + emb_dim   # <-- concat inputs + embedding

        for i, out_ch in enumerate(channels):
            dilation = 2**i
            layers.append(TemporalBlock(in_ch, out_ch, kernel, dilation, dropout))
            in_ch = out_ch

        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(channels[-1], 1)

    def forward(self, x, tic_ids):
        # x: (B, L, F), tic_ids: (B)
        emb = self.emb(tic_ids)                  # (B, emb_dim)
        emb = emb.unsqueeze(1).repeat(1, x.size(1), 1)  # (B, L, emb_dim)

        x = torch.cat([x, emb], dim=2)           # (B, L, F+emb_dim)
        x = x.transpose(1, 2)                    # (B, C, L)

        y = self.tcn(x)                          # (B, C, L)
        y = y.mean(dim=2)                        # GAP
        return self.fc(y).squeeze(-1)

model_te = TCNwithTicker(
    input_dim=len(tcn_features),
    num_tickers=num_tickers,
    emb_dim=12,                   # bigger for more power
    channels=[64,64,64],
    kernel=3,
    dropout=0.2
).to(device)

criterion_te = nn.MSELoss()
optimizer_te = torch.optim.Adam(model_te.parameters(), lr=1e-3)

# -------------------------------------------------------
# 6. Train + Early stopping
# -------------------------------------------------------
best_val = float("inf")
pat = 0

for epoch in range(1, 21):
    # ---- TRAIN ----
    model_te.train()
    train_loss = 0

    for xb, yb, ticb in train_loader_te:
        xb, yb, ticb = xb.to(device), yb.to(device), ticb.to(device)
        optimizer_te.zero_grad()
        preds = model_te(xb, ticb)
        loss = criterion_te(preds, yb)
        loss.backward()
        optimizer_te.step()
        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader_te.dataset)

    # ---- VALID ----
    model_te.eval()
    val_loss = 0
    with torch.no_grad():
        for xb, yb, ticb in val_loader_te:
            xb, yb, ticb = xb.to(device), yb.to(device), ticb.to(device)
            preds = model_te(xb, ticb)
            val_loss += criterion_te(preds, yb).item() * xb.size(0)

    val_loss /= len(val_loader_te.dataset)

    print(f"Epoch {epoch:02d} | Train MSE={train_loss:.6e} | Val MSE={val_loss:.6e}")

    if val_loss + 1e-6 < best_val:
        best_val = val_loss
        best_state = model_te.state_dict()
        pat = 0
    else:
        pat += 1
        if pat >= 3:
            print("Early stopping.")
            break

model_te.load_state_dict(best_state)

# -------------------------------------------------------
# 7. TEST EVALUATION
# -------------------------------------------------------
model_te.eval()
preds = []
true  = []

with torch.no_grad():
    for xb, yb, ticb in test_loader_te:
        xb, yb, ticb = xb.to(device), yb.to(device), ticb.to(device)
        preds.append(model_te(xb, ticb).cpu().numpy())
        true.append(yb.cpu().numpy())

y_pred_te = np.concatenate(preds)
y_true_te = np.concatenate(true)

te_rmse = np.sqrt(mean_squared_error(y_true_te, y_pred_te))
te_da   = (np.sign(y_pred_te) == np.sign(y_true_te)).mean()
te_ic   = spearmanr(y_pred_te, y_true_te).correlation

print("\n==============================")
print("     TCN + TICKER EMBEDDINGS")
print("==============================")
print("Test RMSE:", te_rmse)
print("Test DA:  ", te_da)
print("Test IC:  ", te_ic)


Using: cuda
Tickers: 496


: 